# Data Collection Pipeline

1. [https://www.ncbi.nlm.nih.gov/datasets/genome/](https://www.ncbi.nlm.nih.gov/datasets/genome/)


In [1]:
import os
import glob
import time
import csv
import requests
from io import StringIO
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

import xml.etree.ElementTree as ET

from Bio import SeqIO
from Bio import Entrez
from Bio.SeqIO.FastaIO import FastaWriter


## Configuration and Schema Definition

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

EMAIL = "axa2273@student.bham.ac.uk"  # Set your email for NCBI API guidelines
API_KEY = None                   # Optional NCBI API Key string

MAX_RETRIES = 3
REQUEST_DELAY = 0.10 if API_KEY else 0.34
RETRY_DELAY = 2.0
BATCH_SIZE = 200
TOP_N = 5
OUTPUT_TSV = "assembly_to_srr.tsv"

# NCBI Entrez Endpoints
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ESUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
ELINK_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

# Platform hierarchy & PacBio HiFi detection rules
PLATFORM_PREFERENCE = [
    "PACBIO_HIFI", "PACBIO_SMRT", "ILLUMINA", "OXFORD_NANOPORE", "ION_TORRENT"
]

TECH_KEYWORD_MAP = {
    "PACBIO_HIFI": ["hifi", "ccs", "circular consensus", "revio", "sequel ii", "sequel 2", "pacbio hifi"],
    "PACBIO_SMRT": ["pacbio", "pacific biosciences", "sequel", "smrt", "rsii", "clr"],
    "ILLUMINA": ["illumina", "hiseq", "miseq", "nextseq", "nova-seq", "novaseq", "solexa"],
    "OXFORD_NANOPORE": ["nanopore", "ont", "minion", "gridion", "promethion"],
    "ION_TORRENT": ["ion torrent", "pgm", "proton"]
}

HIFI_INDICATORS = {
    "models": ["revio", "sequel ii", "sequel 2", "onsemble"],
    "keywords": ["hifi", "ccs", "circular consensus"]
}

essential_columns = [
     # === SEQUENCING DATA ===
    'SRR ID',                          # Link to raw FASTQ reads
    
    # === IDENTIFICATION & LINKING ===
    'Assembly Accession',              # Primary identifier for downloading genomes
    'Assembly BioSample Accession',    # Link to metadata and SRA
    'Assembly BioProject Accession',   # Link to sequencing project
    'Organism Name',                   # Species confirmation (E. coli vs K. pneumoniae)
    
    # === STRAIN IDENTIFICATION ===
    'Organism Infraspecific Names Strain',   # Strain name
    
    # === ASSEMBLY QUALITY ===
    'Assembly Level',                  # Must be "Complete Genome"
    'Assembly Stats Number of Contigs', # Indicates plasmid count (chromosome + plasmids)
    'Assembly Stats Total Sequence Length', # Genome size validation
    'Assembly Stats Contig N50',       # Assembly quality metric
    
    # === QUALITY CONTROL ===
    'CheckM completeness',             # Must be >90%
    'CheckM contamination',            # Must be <5%
    
    # === PLASMID DETECTION (if available) ===
    'Assembly Stats Total Number of Chromosomes', # Replicon count
    
    # === SEQUENCING PLATFORM ===
    'Assembly Sequencing Tech',        # ONT, PacBio identification
    
    # === CLINICAL/GEOGRAPHIC METADATA ===
    'Host',                            # Must be "Homo sapiens"
    'Isolation_Source',                # Clinical specimen type
    'Geo_Location',                    # Geographic diversity
    'Collection_Date',                 # Temporal tracking
]


## Helper Functions

In [ ]:
def extract_biosample_metadata(biosamples, output_path='ecoli_metadata.tsv'):
    print(f"First 10 accessions: {biosamples[:10]}")
    # specify an email
    Entrez.email = "axa2273@student.bham.ac.uk"  # Required by NCBI
    
    # Output file
    output = open(output_path, 'w')
    output.write('BioSample\tOrganism\tStrain\tIsolation_Source\tGeo_Location\tHost\tCollection_Date\n')
    
    for i, biosample in enumerate(biosamples):
        #print(f"{i+1}. {biosample} Completed.")
        if i % 100 == 0:
            print(f"Completed {i+1}")
        try:
            # Fetch BioSample record
            handle = Entrez.efetch(db="biosample", id=biosample, rettype="xml")
            record = handle.read()
            handle.close()
            
            # Parse XML (simplified - you may need to parse more carefully)
            import xml.etree.ElementTree as ET
            root = ET.fromstring(record)
            
            organism = root.find('.//OrganismName').text if root.find('.//OrganismName') is not None else 'N/A'
            
            # Extract attributes
            attributes = {}
            for attr in root.findall('.//Attribute'):
                attr_name = attr.get('attribute_name')
                attr_value = attr.text
                attributes[attr_name] = attr_value
            
            strain = attributes.get('strain', 'N/A')
            isolation = attributes.get('isolation_source', 'N/A')
            location = attributes.get('geo_loc_name', 'N/A')
            host = attributes.get('host', 'N/A')
            date = attributes.get('collection_date', 'N/A')

            print(f'{biosample}\t{organism}\t{strain}\t{isolation}\t{location}\t{host}\t{date}\n')
            output.write(f'{biosample}\t{organism}\t{strain}\t{isolation}\t{location}\t{host}\t{date}\n')
            
            time.sleep(0.4)  # NCBI rate limit: 3 requests/second
            
        except Exception as e:
            print(f"Error processing {biosample}: {e}")
    
    output.close()
    print("Metadata extraction complete!")


def classify_long_read_platform(x):
    if pd.isna(x):
        return np.nan
    
    x = x.lower()
    
    # Detect platforms
    has_pacbio = any(term in x for term in [
        "pacbio", "pacific biosciences", "smrt", "pac bio", "sequel", "hifi", "ccs", "clr", "revio"
    ])
    
    has_ont = any(term in x for term in [
        "nanopore", "oxford", "minion", "gridion", "promethion", "ont"
    ])
    
    # Priority rule:
    # If both present, classify based on dominant long-read technology
    if has_pacbio and not has_ont:
        return "PacBio"
    
    if has_ont and not has_pacbio:
        return "Oxford Nanopore"
    
    if has_pacbio and has_ont:
        # Rare case – mixed long-read platforms
        # You may change logic if needed
        return "PB_ONT"
    
    # If no long-read platform detected
    return np.nan


def qc_filtering(df, metadata, 
                 bs_col='Assembly BioSample Accession', 
                 checkM_filtering=True,
                 filtered_output_path="data/ecoli_assemblies_with_human_isolates_filtered.txt"):
    print(f"Initial assemblies: {df.shape[0]}")
    print(f"Metadata records: {metadata.shape[0]}")
    
    # === FILTER 1: Human host ===
    human_host = metadata['Host'].str.contains(
        "Homo sapiens|human", 
        case=False, 
        na=False, 
        regex=True
    )
    
    # === FILTER 2: Clinical isolation source ===
    # Accept clinical specimens
    clinical_keywords = r'blood|urine|wound|sputum|respiratory|clinical|patient|hospital|CSF|cerebrospinal|gut|intestin|colon|urinary|bladder'
    clinical_source = metadata['Isolation_Source'].str.contains(
        clinical_keywords,
        case=False,
        na=False,
        regex=True
    )
    
    # Reject environmental sources even if host=human
    environmental_keywords = r'sewage|wastewater|food|environment(?!al sample)|water(?! sample)|soil|river|lake'
    environmental_source = metadata['Isolation_Source'].str.contains(
        environmental_keywords,
        case=False,
        na=False,
        regex=True
    )
    
    # === COMBINED FILTER ===
    # Must have human host AND (clinical source OR no environmental source)
    valid_isolates = human_host & (~environmental_source | clinical_source)
    
    metadata_filtered = metadata[valid_isolates].copy().rename(columns={"BioSample": bs_col})
    
    print(f"\nAfter human host filter: {human_host.sum()}")
    print(f"After environmental exclusion: {valid_isolates.sum()}")
    
    # === MERGE ===
    df = df.merge(metadata_filtered, on=bs_col, how='inner')
    print(f"\nAfter merge: {df.shape[0]}")

    if not checkM_filtering:
        return df
    
    # Convert to numeric safely (handles strings like "98.7")
    df["CheckM completeness"] = pd.to_numeric(df["CheckM completeness"], errors="coerce")
    df["CheckM contamination"] = pd.to_numeric(df["CheckM contamination"], errors="coerce")
    
    # Explicitly drop rows with missing QC values
    df_qc = df.dropna(subset=["CheckM completeness","CheckM contamination"]).copy()
    
    # Apply thresholds
    df_qc = df_qc[(df_qc["CheckM completeness"] > 95) & (df_qc["CheckM contamination"] < 5)]
    
    print("Assemblies after QC filtering:", len(df_qc))
    
    # Create standardized long-read platform column
    df_qc["Long_Read_Platform"] = df_qc["Assembly Sequencing Tech"].apply(classify_long_read_platform)
    
    # Keep only rows with long-read technology (including hybrid)
    df_final = df_qc[df_qc["Long_Read_Platform"].notna()].copy()
    
    print("Total assemblies after long-read filtering:", len(df_final))
    #df_final.to_csv("data/ecoli_assemblies_with_human_isolates_filtered.txt", sep="\t", index=False)
    print("Filtered Data Saved to", filtered_output_path)
    return df_final

def get_srr(df, n=TOP_N, term='PACBIO', limit=300):
    columns = [
        'Assembly Accession',
        'Assembly BioSample Accession',
        'Assembly BioProject Accession'
    ]
    
    srr_list = []
    term = term.upper()
    
    for i in range(1, n + 1):
        platform_col = f'srr_{i}_platform'
        srr_col = f'srr_{i}'
        
        if platform_col not in df.columns or srr_col not in df.columns:
            continue
        
        mask = df[platform_col].astype(str).str.upper().str.startswith(term)
        
        subset = df.loc[mask, columns + [srr_col]].values.tolist()
        
        for row in subset:
            srr_list.append(tuple(row))
            if len(srr_list) >= limit:
                return srr_list
    
    return srr_list


In [4]:
# ── API Helpers ───────────────────────────────────────────────────────────────

def ncbi_get(url: str, params: dict, timeout: int = 30) -> Optional[requests.Response]:
    """GET request wrapper with retry logic for network drops."""
    params = {**params, "email": EMAIL}
    if API_KEY:
        params["api_key"] = API_KEY

    for attempt in range(1, MAX_RETRIES + 1):
        time.sleep(REQUEST_DELAY)
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    return None

def gcf_to_assembly_uid(gcf: str) -> Optional[str]:
    r = ncbi_get(ESEARCH_URL, {"db": "assembly", "term": f"{gcf}[Assembly Accession]", "retmode": "json"})
    if not r: return None
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    return ids[0] if ids else None

def assembly_uid_to_sra_uids(assembly_uid: str) -> List[str]:
    r = ncbi_get(ELINK_URL, {"dbfrom": "assembly", "db": "sra", "id": assembly_uid, "retmode": "json"})
    if not r: return []
    try:
        linksets = r.json().get("linksets", [])
        if linksets:
            for ldb in linksets[0].get("linksetdbs", []):
                if ldb.get("linkname") == "assembly_sra":
                    return [str(uid) for uid in ldb.get("links", [])]
    except Exception:
        pass
    return []

def biosample_to_sra_uids(biosample: str) -> List[str]:
    r = ncbi_get(ESEARCH_URL, {"db": "sra", "term": f"{biosample}[BioSample]", "retmax": 1000, "retmode": "json"})
    if not r: return []
    return r.json().get("esearchresult", {}).get("idlist", [])

def fetch_biosample_from_assembly(assembly_uid: str) -> Optional[str]:
    r = ncbi_get(ESUMMARY_URL, {"db": "assembly", "id": assembly_uid, "retmode": "json"})
    if not r: return None
    doc = r.json().get("result", {}).get(assembly_uid, {})
    return doc.get("biosampleaccn", "").strip() or None

def uids_to_runinfo(uids: List[str]) -> List[Dict]:
    rows = []
    for i in range(0, len(uids), BATCH_SIZE):
        batch = uids[i : i + BATCH_SIZE]
        r = ncbi_get(EFETCH_URL, {"db": "sra", "id": ",".join(batch), "rettype": "runinfo", "retmode": "text"}, timeout=60)
        if not r or not r.text.strip(): continue
        reader = csv.DictReader(StringIO(r.text))
        for row in reader:
            srr = row.get("Run", "").strip()
            if srr.startswith(("SRR", "ERR", "DRR")):
                rows.append(row)
    return rows

# ── Platform Detection & Ranking ──────────────────────────────────────────────

def is_pacbio_hifi(runinfo_row: Dict) -> bool:
    if runinfo_row.get("Platform", "").upper() != "PACBIO_SMRT":
        return False
    model = str(runinfo_row.get("Model", "")).lower()
    if any(m in model for m in HIFI_INDICATORS["models"]):
        return True
    combined = " ".join([str(runinfo_row.get(k, "")) for k in ["Model", "Experiment", "SampleName", "LibraryName"]]).lower()
    return any(kw in combined for kw in HIFI_INDICATORS["keywords"])

def resolve_run_platform(runinfo_row: Dict) -> str:
    raw_platform = runinfo_row.get("Platform", "").upper()
    if raw_platform == "PACBIO_SMRT" and is_pacbio_hifi(runinfo_row):
        return "PACBIO_HIFI"
    return raw_platform

def infer_expected_platform(seq_tech: str) -> Optional[str]:
    if not seq_tech or pd.isna(seq_tech): return None
    tech_lower = str(seq_tech).lower()
    for platform, keywords in TECH_KEYWORD_MAP.items():
        if any(kw in tech_lower for kw in keywords):
            return platform
    return None

def rank_key(row: Dict, expected_platform: Optional[str]) -> tuple:
    detected_platform = resolve_run_platform(row)
    expected_miss = 0 if (expected_platform and detected_platform == expected_platform) else 1
    try: plat_score = PLATFORM_PREFERENCE.index(detected_platform)
    except ValueError: plat_score = len(PLATFORM_PREFERENCE)
    try: bases = int(row.get("bases", 0) or 0)
    except ValueError: bases = 0
    return (expected_miss, plat_score, -bases)

def select_top_n_srrs(runs: List[Dict], expected_platform: Optional[str] = None, n: int = TOP_N) -> List[Dict]:
    if not runs: return []
    genomic = [r for r in runs if r.get("LibrarySource", "").upper() == "GENOMIC"]
    pool = genomic if genomic else runs
    wgs = [r for r in pool if r.get("LibraryStrategy", "").upper() == "WGS"]
    pool = wgs if wgs else pool
    pool.sort(key=lambda r: rank_key(r, expected_platform))
    return pool[:n]

# ── Pipeline & Atomic Persistence ─────────────────────────────────────────────

def _empty_result(row: pd.Series) -> Dict:
    base = {
        "Assembly Accession":            row.get("Assembly Accession", ""),
        "Assembly BioSample Accession":  row.get("Assembly BioSample Accession", ""),
        "Assembly BioProject Accession": row.get("Assembly BioProject Accession", ""),
        "Assembly Sequencing Tech":      row.get("Assembly Sequencing Tech", ""),
        "expected_platform":             None,
        "n_sra_runs":                    0,
        "status":                        "pending",
        "platform_match_srr1":           None,
    }
    for i in range(1, TOP_N + 1):
        base[f"srr_{i}"]          = None
        base[f"srr_{i}_platform"] = None
        base[f"srr_{i}_is_hifi"]  = None
        base[f"srr_{i}_model"]    = None
        base[f"srr_{i}_bases"]    = None
    return base

def process_assembly_row(row: pd.Series) -> Dict:
    result = _empty_result(row)
    gcf = str(row.get("Assembly Accession", "")).strip()
    biosample = str(row.get("Assembly BioSample Accession", "")).strip()
    seq_tech = row.get("Assembly Sequencing Tech", "")

    expected_platform = infer_expected_platform(seq_tech)
    result["expected_platform"] = expected_platform

    if not gcf or gcf in ("nan", "None", ""):
        result["status"] = "missing_gcf"
        return result

    assembly_uid = gcf_to_assembly_uid(gcf)
    sra_uids = assembly_uid_to_sra_uids(assembly_uid) if assembly_uid else []

    if not sra_uids:
        if (not biosample or biosample in ("nan", "None", "")) and assembly_uid:
            biosample = fetch_biosample_from_assembly(assembly_uid)
            result["Assembly BioSample Accession"] = biosample

        if biosample and biosample not in ("nan", "None", ""):
            sra_uids = biosample_to_sra_uids(biosample)

    if not sra_uids:
        result["status"] = "no_sra_runs"
        return result

    runs = uids_to_runinfo(sra_uids)
    result["n_sra_runs"] = len(runs)
    if not runs:
        result["status"] = "no_runinfo"
        return result

    top_runs = select_top_n_srrs(runs, expected_platform=expected_platform, n=TOP_N)
    if not top_runs:
        result["status"] = "no_suitable_run"
        return result

    for i, run in enumerate(top_runs, 1):
        detected_platform = resolve_run_platform(run)
        result[f"srr_{i}"]          = run.get("Run")
        result[f"srr_{i}_platform"] = detected_platform
        result[f"srr_{i}_is_hifi"]  = (detected_platform == "PACBIO_HIFI")
        result[f"srr_{i}_model"]    = run.get("Model")
        result[f"srr_{i}_bases"]    = run.get("bases")

    best_platform = resolve_run_platform(top_runs[0])
    result["platform_match_srr1"] = (
        "yes" if expected_platform and best_platform == expected_platform else
        "no" if expected_platform else "unknown"
    )
    result["status"] = "ok"
    return result

def write_single_row_to_file(rec: Dict, output_tsv: str):
    """Appends one record to file immediately to prevent loss on interruption."""
    meta_cols = [
        "Assembly Accession", "Assembly BioSample Accession",
        "Assembly BioProject Accession", "Assembly Sequencing Tech",
        "expected_platform", "n_sra_runs", "status", "platform_match_srr1"
    ]
    srr_cols = [
        f"srr_{i}{suffix}"
        for i in range(1, TOP_N + 1)
        for suffix in ("", "_platform", "_is_hifi", "_model", "_bases")
    ]
    all_cols = meta_cols + srr_cols

    row_df = pd.DataFrame([rec])[all_cols]
    file_exists = os.path.exists(output_tsv) and os.path.getsize(output_tsv) > 0
    row_df.to_csv(output_tsv, sep="\t", index=False, mode="a" if file_exists else "w", header=not file_exists)

def link_assemblies_to_srr(df: pd.DataFrame, output_tsv: str = OUTPUT_TSV) -> pd.DataFrame:
    """Processes DataFrame with zero data loss on interrupt and automatic resume."""
    completed_gcfs = set()
    if os.path.exists(output_tsv) and os.path.getsize(output_tsv) > 0:
        try:
            existing_df = pd.read_csv(output_tsv, sep="\t", usecols=["Assembly Accession"])
            completed_gcfs = set(existing_df["Assembly Accession"].dropna().astype(str).str.strip())
            print(f"[*] Resuming: Found {len(completed_gcfs)} assemblies already processed.")
        except Exception:
            print("[!] Found existing output file but couldn't parse it. Appending fresh entries.")

    total = len(df)
    print(f"Processing {total} assemblies...\n" + "─" * 60)

    for idx, (_, row) in enumerate(df.iterrows(), 1):
        gcf = str(row.get("Assembly Accession", "")).strip()

        if gcf in completed_gcfs:
            print(f"[{idx:>{len(str(total))}}/{total}] {gcf:<15} | Skipped (Already Processed)")
            continue

        try:
            rec = process_assembly_row(row)
            write_single_row_to_file(rec, output_tsv)
            completed_gcfs.add(gcf)

            srr_ids = [rec[f"srr_{i}"] for i in range(1, TOP_N + 1) if rec[f"srr_{i}"]]
            print(f"[{idx:>{len(str(total))}}/{total}] {gcf:<15} | Status: {rec['status']:<12} | Top SRRs: {srr_ids}")
        except KeyboardInterrupt:
            print("\n[!] Interrupt detected! All completed assemblies up to this point are saved.")
            break
        except Exception as e:
            print(f"[{idx:>{len(str(total))}}/{total}] {gcf:<15} | Error: {e}")
            continue

    print(f"\n[✓] Results flushed directly to disk: {output_tsv}")
    return pd.read_csv(output_tsv, sep="\t")


In [5]:
# ── NCBI API helpers ──────────────────────────────────────────────────────────

def ncbi_get(url: str, params: dict, timeout: int = 30) -> Optional[requests.Response]:
    """GET request to any NCBI Entrez endpoint with retry logic."""
    params = {**params, "email": EMAIL}
    if API_KEY:
        params["api_key"] = API_KEY

    for attempt in range(1, MAX_RETRIES + 1):
        time.sleep(REQUEST_DELAY)
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as exc:
            print(f"      [attempt {attempt}/{MAX_RETRIES}] {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    return None


def biosample_to_sra_uids(biosample: str) -> List[str]:
    """BioSample accession → list of SRA UIDs."""
    r = ncbi_get(ESEARCH_URL, {
        "db":      "sra",
        "term":    f"{biosample}[BioSample]",
        "retmax":  1000,
        "retmode": "json",
    })
    if not r:
        return []
    return r.json().get("esearchresult", {}).get("idlist", [])


def uids_to_runinfo(uids: List[str]) -> List[Dict]:
    """SRA UIDs → list of runinfo row dicts (batched to avoid HTTP 400)."""
    rows = []
    for i in range(0, len(uids), BATCH_SIZE):
        batch = uids[i : i + BATCH_SIZE]
        r = ncbi_get(EFETCH_URL, {
            "db":      "sra",
            "id":      ",".join(batch),
            "rettype": "runinfo",
            "retmode": "text",
        }, timeout=60)
        if not r or not r.text.strip():
            continue
        reader = csv.DictReader(StringIO(r.text))
        for row in reader:
            srr = row.get("Run", "").strip()
            if srr.startswith(("SRR", "ERR", "DRR")):
                rows.append(row)
    return rows


def gcf_to_biosample_via_api(gcf: str) -> Optional[str]:
    """
    Fallback when BioSample column is empty.
    GCF → Assembly UID → BioSample accession (via esummary, single extra call).
    """
    # Step A: GCF accession → Assembly UID
    r = ncbi_get(ESEARCH_URL, {"db": "assembly", "term": gcf, "retmode": "json"})
    if not r:
        return None
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    if not ids:
        return None

    # Step B: Assembly UID → BioSample (stored directly in esummary JSON)
    uid = ids[0]
    r = ncbi_get(ESUMMARY_URL, {"db": "assembly", "id": uid, "retmode": "json"})
    if not r:
        return None
    doc = r.json().get("result", {}).get(uid, {})
    bs  = doc.get("biosampleaccn", "").strip()
    return bs or None


# ── Run selection logic ───────────────────────────────────────────────────────

def infer_expected_platform(seq_tech: str) -> Optional[str]:
    """
    Map free-text 'Assembly Sequencing Tech' value to an NCBI platform string.
    Returns None if the field is empty or unrecognised.
    """
    if not seq_tech or pd.isna(seq_tech):
        return None
    tech_lower = str(seq_tech).lower()
    for platform, keywords in TECH_KEYWORD_MAP.items():
        if any(kw in tech_lower for kw in keywords):
            return platform
    return None


def rank_key(row: Dict, expected_platform: Optional[str]) -> tuple:
    """
    Sorting key for a runinfo row (lower = better).
    """
    platform = row.get("Platform", "").upper()

    try:
        plat_score = PLATFORM_PREFERENCE.index(platform)
    except ValueError:
        plat_score = len(PLATFORM_PREFERENCE)

    expected_miss = 0 if (expected_platform and platform == expected_platform) else 1

    try:
        bases = int(row.get("bases", 0) or 0)
    except ValueError:
        bases = 0

    return (expected_miss, plat_score, -bases)


def select_top_n_srrs(runs: List[Dict],
                      expected_platform: Optional[str] = None,
                      n: int = TOP_N) -> List[Dict]:
    """
    Filter and rank runinfo rows, returning the top-n best runs.
    """
    if not runs:
        return []

    # Filter 1: prefer GENOMIC source
    genomic = [r for r in runs if r.get("LibrarySource", "").upper() == "GENOMIC"]
    pool    = genomic if genomic else runs

    # Filter 2: prefer WGS strategy
    wgs  = [r for r in pool if r.get("LibraryStrategy", "").upper() == "WGS"]
    pool = wgs if wgs else pool

    pool.sort(key=lambda r: rank_key(r, expected_platform))
    return pool[:n]


# ── Per-assembly pipeline ─────────────────────────────────────────────────────

def _empty_result(row: pd.Series) -> Dict:
    """
    Initialise a result dict with all output columns set to None.
    """
    base = {
        "Assembly Accession":            row.get("Assembly Accession", ""),
        "Assembly BioSample Accession":  row.get("Assembly BioSample Accession", ""),
        "Assembly BioProject Accession": row.get("Assembly BioProject Accession", ""),
        "Assembly Sequencing Tech":      row.get("Assembly Sequencing Tech", ""),
        "expected_platform":             None,
        "n_sra_runs":                    0,
        "status":                        "pending",
        "platform_match_srr1":           None,
    }
    for i in range(1, TOP_N + 1):
        base[f"srr_{i}"]          = None
        base[f"srr_{i}_platform"] = None
        base[f"srr_{i}_model"]    = None
        base[f"srr_{i}_bases"]    = None
    return base


def process_assembly_row(row: pd.Series) -> Dict:
    """
    Full pipeline for one assembly row from ecoli_df.
    """
    result   = _empty_result(row)
    gcf      = str(row.get("Assembly Accession", "")).strip()
    biosample = str(row.get("Assembly BioSample Accession", "")).strip()
    seq_tech  = row.get("Assembly Sequencing Tech", "")

    # Infer expected sequencing platform from metadata
    expected_platform          = infer_expected_platform(seq_tech)
    result["expected_platform"] = expected_platform

    # ── BioSample: use column value or fall back to API lookup ──
    if not biosample or biosample in ("nan", "None", ""):
        print(f"    BioSample missing for {gcf} — querying NCBI Assembly db …")
        biosample = gcf_to_biosample_via_api(gcf)
        if not biosample:
            result["status"] = "no_biosample"
            return result
        result["Assembly BioSample Accession"] = biosample

    # ── BioSample → SRA UIDs ──
    uids = biosample_to_sra_uids(biosample)
    if not uids:
        result["status"] = "no_sra_runs"
        return result

    # ── UIDs → RunInfo ──
    runs = uids_to_runinfo(uids)
    result["n_sra_runs"] = len(runs)
    if not runs:
        result["status"] = "no_runinfo"
        return result

    # ── Select top-N SRRs ──
    top_runs = select_top_n_srrs(runs, expected_platform=expected_platform, n=TOP_N)
    if not top_runs:
        result["status"] = "no_suitable_run"
        return result

    for i, run in enumerate(top_runs, 1):
        result[f"srr_{i}"]          = run.get("Run")
        result[f"srr_{i}_platform"] = run.get("Platform", "").upper()
        result[f"srr_{i}_model"]    = run.get("Model")
        result[f"srr_{i}_bases"]    = run.get("bases")

    # Convenience: did the best SRR platform match our metadata?
    best_platform = top_runs[0].get("Platform", "").upper()
    result["platform_match_srr1"] = (
        "yes"     if expected_platform and best_platform == expected_platform else
        "no"      if expected_platform else
        "unknown"
    )
    result["status"] = "ok"
    return result


# ── Batch runner ──────────────────────────────────────────────────────────────

def link_assemblies_to_srr(ecoli_df: pd.DataFrame,
                            output_tsv: str = "ecoli_gcf_to_srr.tsv") -> pd.DataFrame:
    """
    Main entry point to link assemblies
    """
    records = []
    total   = len(ecoli_df)

    print(f"Processing {total} assemblies …\n{'─' * 65}")

    for idx, (_, row) in enumerate(ecoli_df.iterrows(), 1):
        gcf = row.get("Assembly Accession", f"row_{idx}")
        print(f"[{idx:>{len(str(total))}}/{total}]  {gcf} … ", end="", flush=True)

        rec = process_assembly_row(row)
        records.append(rec)

        # Compact status line
        srr_ids = [rec[f"srr_{i}"] for i in range(1, TOP_N + 1) if rec[f"srr_{i}"]]
        print(
            f"{rec['status']:<22}  "
            f"platform_match={rec['platform_match_srr1']}  "
            f"runs_found={rec['n_sra_runs']}  "
            f"top_srrs={srr_ids}"
        )

    results_df = pd.DataFrame(records)

    # ── Column ordering: metadata first, then srr_1…srr_5 blocks ──
    meta_cols = [
        "Assembly Accession", "Assembly BioSample Accession",
        "Assembly BioProject Accession", "Assembly Sequencing Tech",
        "expected_platform", "n_sra_runs", "status", "platform_match_srr1",
    ]
    srr_cols = [
        f"srr_{i}{suffix}"
        for i in range(1, TOP_N + 1)
        for suffix in ("", "_platform", "_model", "_bases")
    ]
    results_df = results_df[meta_cols + srr_cols]
    results_df.to_csv(output_tsv, sep="\t", index=False)

    # ── Summary ──
    ok        = (results_df.status == "ok").sum()
    match_yes = (results_df.platform_match_srr1 == "yes").sum()
    match_no  = (results_df.platform_match_srr1 == "no").sum()

    print(f"\n{'═' * 65}")
    print(f"  Total assemblies processed : {total}")
    print(f"  Successfully linked        : {ok}")
    print(f"  Platform match  (yes)      : {match_yes}")
    print(f"  Platform mismatch (no)     : {match_no}")
    print(f"  No SRA runs found          : {(results_df.status == 'no_sra_runs').sum()}")
    print(f"  No BioSample               : {(results_df.status == 'no_biosample').sum()}")
    print(f"  Output saved to            : {output_tsv}")
    print(f"{'═' * 65}\n")

    # Flag mismatches for manual review
    mismatches = results_df[results_df.platform_match_srr1 == "no"]
    if not mismatches.empty:
        print(f"  ⚠  {len(mismatches)} platform mismatches — review these rows:")
        review_cols = [
            "Assembly Accession", "Assembly Sequencing Tech",
            "expected_platform", "srr_1_platform", "srr_1",
        ]
        print(mismatches[review_cols].to_string(index=False))

    return results_df

def get_best_samples(df, ECOLI_GENOME_SIZE=5_000_000, MIN_COVERAGE=30):
    # 1. Clean data & convert numeric columns
    df['bases'] = pd.to_numeric(df['bases'], errors='coerce')
    df['avgLength'] = pd.to_numeric(df['avgLength'], errors='coerce')
    df['ReleaseDate'] = pd.to_datetime(df['ReleaseDate'], errors='coerce')
    
    # 2. Calculate sequencing depth
    df['coverage'] = df['bases'] / ECOLI_GENOME_SIZE
    
    # 3. Apply Quality Filter: 
    #    - At least 30x coverage
    #    - Modern read length (e.g., avgLength >= 100 for decent short-read Illumina data)
    quality_samples = df[
        (df['coverage'] >= MIN_COVERAGE) & 
        (df['avgLength'] >= 100)
    ]
    
    # 4. Rank by metadata completeness AND highest coverage
    best_samples = quality_samples.sort_values(
        by=['coverage', 'ReleaseDate'], 
        ascending=[False, False]
    )
    return best_samples.iloc[:400]


In [ ]:
def extract_biosample_metadata(
    biosamples,
    output_path="biosample_metadata.tsv",
    email="axa2273@student.bham.ac.uk",
    sleep_time=0.35,
):
    """
    Download BioSample metadata from NCBI.

    Features
    --------
    * Resume interrupted runs
    * Safe XML parsing
    * Automatic progress reporting
    * Extracts common BioSample metadata
    * Stores selected BioSample attributes
    """

    Entrez.email = email

    # ------------------------------------------------------------
    # Resume previously processed samples
    # ------------------------------------------------------------

    processed = set()

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            next(f, None)

            for line in f:
                if line.strip():
                    processed.add(line.split("\t", 1)[0])

        print(f"Found {len(processed)} previously processed BioSamples.")

    # ------------------------------------------------------------
    # BioSample attributes to export
    # ------------------------------------------------------------
    
    attribute_fields = [
        "strain",
        "genotype",
        "host",
        "host_disease",
        "isolation_source",
        "collection_date",
        "geo_loc_name",
    ]
    
    header = [
        "BioSample",
        "Organism",
        "TaxID",
        "BioProject",
        "SRA",
        "Status",
    ] + attribute_fields
    
    write_header = (
        not os.path.exists(output_path)
        or os.path.getsize(output_path) == 0
    )
    
    total = len(biosamples)
    
    with open(output_path, "a", encoding="utf-8") as out:

        if write_header:
            out.write("\t".join(header) + "\n")

        for i, biosample in enumerate(biosamples, start=1):

            if biosample in processed:
                continue

            if i == 1 or i % 100 == 0:
                print(f"[{i}/{total}] {biosample}")

            try:

                # ------------------------------------------------
                # Download XML
                # ------------------------------------------------

                with Entrez.efetch(
                    db="biosample",
                    id=biosample,
                    retmode="xml",
                ) as handle:

                    xml_text = handle.read()

                root = ET.fromstring(xml_text)

                biosample_node = root.find("BioSample")

                if biosample_node is None:
                    raise RuntimeError("BioSample node not found.")

                # ------------------------------------------------
                # Organism
                # ------------------------------------------------

                organism_node = biosample_node.find(".//Organism")

                organism = "N/A"
                taxid = "N/A"

                if organism_node is not None:

                    taxid = organism_node.get("taxonomy_id", "N/A")

                    organism = (
                        organism_node.get("taxonomy_name")
                        or organism_node.findtext("OrganismName")
                        or "N/A"
                    )

                # ------------------------------------------------
                # Simple nodes
                # ------------------------------------------------

                title = biosample_node.findtext(
                    "./Description/Title",
                    default="N/A",
                )

                owner = biosample_node.findtext(
                    "./Owner/Name",
                    default="N/A",
                )

                package = biosample_node.findtext(
                    "./Package",
                    default="N/A",
                )

                model = biosample_node.findtext(
                    "./Models/Model",
                    default="N/A",
                )

                # ------------------------------------------------
                # Dates / status
                # ------------------------------------------------

                submission_date = biosample_node.get(
                    "submission_date",
                    "N/A",
                )

                publication_date = biosample_node.get(
                    "publication_date",
                    "N/A",
                )

                last_update = biosample_node.get(
                    "last_update",
                    "N/A",
                )

                status_node = biosample_node.find("./Status")

                status = (
                    status_node.get("status", "N/A")
                    if status_node is not None
                    else "N/A"
                )

                # ------------------------------------------------
                # IDs
                # ------------------------------------------------

                ids = {}

                for node in biosample_node.findall("./Ids/Id"):

                    db = node.get("db")

                    if db and node.text:
                        ids[db] = node.text.strip()

                sra = ids.get("SRA", "N/A")

                # ------------------------------------------------
                # BioProject
                # ------------------------------------------------

                bioproject = "N/A"

                for link in biosample_node.findall("./Links/Link"):

                    if link.get("target") == "bioproject":

                        bioproject = (
                            link.get("label")
                            or (link.text or "").strip()
                            or "N/A"
                        )

                        break

                # ------------------------------------------------
                # Attributes
                # ------------------------------------------------

                attrs = {}

                for attr in biosample_node.findall(".//Attribute"):

                    key = (
                        attr.get("attribute_name", "")
                        .strip()
                        .lower()
                    )

                    value = (attr.text or "").strip()

                    if key:
                        attrs[key] = value

                # ------------------------------------------------
                # Build output row
                # ------------------------------------------------
                
                row = [
                    biosample,
                    organism,
                    taxid,
                    bioproject,
                    sra,
                    status,
                ]
                
                row.extend(
                    attrs.get(field, "N/A")
                    for field in attribute_fields
                )

                clean_row = [str(val).replace("\t", " ").replace("\n", " ").replace("\r", " ") for val in row]
                out.write("\t".join(clean_row) + "\n")
                out.flush()

            except Exception as e:

                print(f"Failed: {biosample}")
                print(f"Reason : {e}")

            time.sleep(sleep_time)

    print("\nFinished.")



In [7]:
ecoli_df = pd.read_csv("data/ecoli_assemblies.txt", sep="\t")
ecoli_df.head()


,Assembly Name,Assembly Accession,Assembly Paired Assembly Accession,Organism Name,Organism Taxonomic ID,ANI Check status,Organism Infraspecific Names Breed,Organism Infraspecific Names Strain,Organism Infraspecific Names Cultivar,Organism Infraspecific Names Ecotype,...,Assembly Submitter,Assembly BioProject Accession,Assembly BioSample Accession,Annotation Count Gene Total,Annotation Count Gene Protein-coding,Annotation Count Gene Pseudogene,Type Material Display Text,CheckM marker set,CheckM completeness,CheckM contamination
0,ASM886v2,GCA_000008865.2,GCF_000008865.2,Escherichia coli O157:H7 str. Sakai,386585,OK,NaN,Sakai substr. RIMD 0509952,NaN,NaN,...,GIRC,PRJNA226,SAMN01911278,5375,5113,136.0,NaN,Escherichia coli,99.51,0.15
1,ASM886v2,GCF_000008865.2,GCA_000008865.2,Escherichia coli O157:H7 str. Sakai,386585,OK,NaN,Sakai substr. RIMD 0509952,NaN,NaN,...,GIRC,PRJNA226,SAMN01911278,5417,5155,136.0,NaN,Escherichia coli,99.51,0.15
2,ASM285371v1,GCA_002853715.1,GCF_002853715.1,Escherichia coli,562,OK,NaN,14EC020,NaN,NaN,...,South China Sea Institute of Oceanology,PRJNA414689,SAMN07807401,5370,4960,289.0,assembly designated as clade exemplar,Escherichia coli,99.43,1.10
3,ASM285371v1,GCF_002853715.1,GCA_002853715.1,Escherichia coli,562,OK,NaN,14EC020,NaN,NaN,...,South China Sea Institute of Oceanology,PRJNA414689,SAMN07807401,5081,4739,219.0,assembly designated as clade exemplar,Escherichia coli,99.43,1.10
4,ASM369716v2,GCA_003697165.2,GCF_003697165.2,Escherichia coli DSM 30083 = JCM 1649 = ATCC 1...,866789,OK,NaN,ATCC 11775,NaN,NaN,...,University of Arkansas for Medical Sciences,PRJNA472652,SAMN10252913,5010,4723,172.0,assembly designated as neotype,Escherichia coli,99.09,0.45


In [8]:
esamples = ecoli_df['Assembly BioSample Accession'].unique().tolist()
with open("data/ecoli_biosamples.txt", "w") as fo:
    fo.write("\n".join(esamples))


In [9]:
#extract_biosample_metadata(esamples, output_path="data/ecoli_metadata.tsv")
ecoli_meta_df = pd.read_csv("data/ecoli_metadata.tsv", sep="\t")
ecoli_meta_df.head()


,BioSample,Organism,Strain,Isolation_Source,Geo_Location,Host,Collection_Date
0,SAMN01911278,Escherichia coli O157:H7 str. Sakai,SAKAI (EHEC),Human intestinal microflora,"Japan: Sakai City, Osaka prefecture",NaN,1996
1,SAMN07807401,Escherichia coli,14EC020,clinical patient,China,NaN,2014
2,SAMN10252913,Escherichia coli DSM 30083 = JCM 1649 = ATCC 1...,ATCC 11775,NaN,NaN,NaN,2018-03-20
3,SAMN08638904,Escherichia coli,NaN,stool,USA,Homo sapiens,1997-08
4,SAMN02911890,Escherichia coli,48,Fecal sample from a deer,Switzerland,NaN,2011-06-01


In [10]:
ecoli_filtered_df = qc_filtering(ecoli_df, ecoli_meta_df)
ecoli_filtered_df.head()


Initial assemblies: 10591
Metadata records: 6298

After human host filter: 2124
After environmental exclusion: 2122

After merge: 3822
Assemblies after QC filtering: 3251
Total assemblies after long-read filtering: 3147
Filtered Data Saved to data/ecoli_assemblies_with_human_isolates_filtered.txt


,Assembly Name,Assembly Accession,Assembly Paired Assembly Accession,Organism Name,Organism Taxonomic ID,ANI Check status,Organism Infraspecific Names Breed,Organism Infraspecific Names Strain,Organism Infraspecific Names Cultivar,Organism Infraspecific Names Ecotype,...,CheckM marker set,CheckM completeness,CheckM contamination,Organism,Strain,Isolation_Source,Geo_Location,Host,Collection_Date,Long_Read_Platform
0,ASM301845v1,GCF_003018455.1,GCA_003018455.1,Escherichia coli,562,OK,NaN,97-3250,NaN,NaN,...,Escherichia coli,99.05,0.77,Escherichia coli,NaN,stool,USA,Homo sapiens,1997-08,PacBio
1,ASM4856894v1,GCA_048568945.1,GCF_048568945.1,Escherichia coli,562,OK,NaN,97_3250,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,97_3250,missing,Missing,Homo sapiens,missing,PacBio
2,ASM4856894v1,GCF_048568945.1,GCA_048568945.1,Escherichia coli,562,OK,NaN,97_3250,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,97_3250,missing,Missing,Homo sapiens,missing,PacBio
3,ASM4857156v1,GCA_048571565.1,GCF_048571565.1,Escherichia coli,562,OK,NaN,10_253,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,10_253,missing,missing,Homo sapiens,missing,PacBio
4,ASM4857156v1,GCF_048571565.1,GCA_048571565.1,Escherichia coli,562,OK,NaN,10_253,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,10_253,missing,missing,Homo sapiens,missing,PacBio


In [11]:
#ecoli_df = pd.read_csv("data/ecoli_assemblies_with_human_isolates_filtered.txt", sep="\t")
ecoli_filtered_df = ecoli_filtered_df[ecoli_filtered_df['Assembly Accession'].apply(lambda x: x.startswith("GCF"))]
ecoli_filtered_df.head()


,Assembly Name,Assembly Accession,Assembly Paired Assembly Accession,Organism Name,Organism Taxonomic ID,ANI Check status,Organism Infraspecific Names Breed,Organism Infraspecific Names Strain,Organism Infraspecific Names Cultivar,Organism Infraspecific Names Ecotype,...,CheckM marker set,CheckM completeness,CheckM contamination,Organism,Strain,Isolation_Source,Geo_Location,Host,Collection_Date,Long_Read_Platform
0,ASM301845v1,GCF_003018455.1,GCA_003018455.1,Escherichia coli,562,OK,NaN,97-3250,NaN,NaN,...,Escherichia coli,99.05,0.77,Escherichia coli,NaN,stool,USA,Homo sapiens,1997-08,PacBio
2,ASM4856894v1,GCF_048568945.1,GCA_048568945.1,Escherichia coli,562,OK,NaN,97_3250,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,97_3250,missing,Missing,Homo sapiens,missing,PacBio
4,ASM4857156v1,GCF_048571565.1,GCA_048571565.1,Escherichia coli,562,OK,NaN,10_253,NaN,NaN,...,Escherichia coli,99.31,0.80,Escherichia coli,10_253,missing,missing,Homo sapiens,missing,PacBio
6,ASM4857164v1,GCF_048571645.1,GCA_048571645.1,Escherichia coli,562,OK,NaN,00_3230,NaN,NaN,...,Escherichia coli,99.31,0.82,Escherichia coli,00_3230,missing,missing,Homo sapiens,missing,PacBio
8,ASM2430068v1,GCF_024300685.1,GCA_024300685.1,Escherichia coli,562,OK,NaN,2003-3014,NaN,NaN,...,Escherichia coli,99.27,0.71,Escherichia coli,2003-3014,NaN,NaN,Homo sapiens,2021,Oxford Nanopore


In [12]:
#link_assemblies_to_srr(ecoli_filtered_df, output_tsv="ecoli_gcf_to_srr.tsv")


In [13]:
#ecoli_link_df = pd.read_csv("data/ecoli_gcf_to_srr.tsv", sep="\t")
#ecoli_link_df


In [14]:
output_path = "data/ecoli_gcf_to_srr_pacbio_only.tsv"
df = pd.read_csv(output_path, sep="\t")

# Dynamically find all srr_N columns that have a corresponding _is_hifi column
srr_indices = [
    col.split("_")[1] 
    for col in df.columns 
    if col.startswith("srr_") and col.endswith("_is_hifi")
]

records = []

for idx in srr_indices:
    srr_col = f"srr_{idx}"
    hifi_col = f"srr_{idx}_is_hifi"
    model_col = f"srr_{idx}_model"
    bases_col = f"srr_{idx}_bases"

    # Select only rows where this specific SRR slot is HiFi
    hifi_subset = df[df[hifi_col] == True].copy()
    
    for _, row in hifi_subset.iterrows():
        records.append({
            "Assembly Accession": row["Assembly Accession"],
            "BioSample": row["Assembly BioSample Accession"],
            "SRR Accession": row[srr_col],
            "Model": row.get(model_col),
            "Bases": row.get(bases_col),
            "Rank_Slot": int(idx)
        })

hifi_df = pd.DataFrame(records).drop_duplicates(subset=["Assembly Accession", "SRR Accession"])
#hifi_df.to_csv("only_hifi_reads.tsv", sep="\t", index=False)
hifi_df.head()


,Assembly Accession,BioSample,SRR Accession,Model,Bases,Rank_Slot
0,GCA_013357365.1,SAMN14449600,SRR12005555,Sequel II,2.288134e+09,1
1,GCF_013357365.1,SAMN14449600,SRR12005555,Sequel II,2.288134e+09,1
2,GCA_009738455.1,SAMN13429994,SRR10598574,Sequel II,5.392916e+09,1
3,GCF_009738455.1,SAMN13429994,SRR10598574,Sequel II,5.392916e+09,1
4,GCA_054908925.1,SAMN54691118,SRR36894546,Sequel IIe,2.515671e+08,1


In [15]:
with open("data/List/ecoli_pacbio_srr_list", "w") as fw:
    fw.write("\n".join(hifi_df["SRR Accession"].unique().tolist()))

with open("data/List/ecoli_pacbio_gcf_list", "w") as fw:
    fw.write("\n".join(hifi_df["Assembly Accession"].unique().tolist()))


## Plasmid Extraction


In [16]:
ref_map = pd.read_csv("data/reference_mapping.tsv", sep="\t")
ref_map.head()


,SRR ID,Assembly Accession,Assembly Accession 2
0,SRR15462540,GCF_013167875.1,GCA_013167875.1
1,SRR15415584,GCF_013167875.1,GCA_013167875.1
2,SRR28579936,GCF_026341465.2,GCA_026341465.2
3,SRR5169293,GCF_002951695.1,GCA_002951695.1
4,SRR10520608,GCF_009664455.1,GCA_009664455.1


In [17]:
with open("data/List/new_ecoli_pacbio_srr_list.txt", "r") as fi:
    new_refs = fi.read().splitlines()

gcf_list = set(ref_map[ref_map['SRR ID'].isin(new_refs)][['Assembly Accession', 'Assembly Accession 2']].values.flatten())
all_genome_paths = glob.glob("data/FinalFASTA/**/*.fna", recursive=True)


In [18]:
# Check whether any reference genome missing for any SRR ID
found = []
not_found = []

for gcf in gcf_list:
    matches = [
        path for path in all_genome_paths
        if gcf in path
    ]
    
    if matches:
        found.append(gcf)
    else:
        not_found.append(gcf)

print("Found:", len(found))
print("Not found:", len(not_found))

print("\nFound GCFs:")
print(found)

print("\nNot found GCFs:")
print(not_found)


Found: 128
Not found: 0

Found GCFs:
['GCF_056784715.1', 'GCA_056784325.1', 'GCA_054902725.1', 'GCA_056784695.1', 'GCA_055389265.1', 'GCF_056784415.1', 'GCF_056784375.1', 'GCF_056784355.1', 'GCA_050957145.1', 'GCA_054903255.1', 'GCA_056784375.1', 'GCA_030295485.1', 'GCF_041233395.1', 'GCA_041344775.1', 'GCF_030295545.1', 'GCF_050957145.1', 'GCA_056784775.1', 'GCF_054903355.1', 'GCA_056784565.1', 'GCA_042192135.1', 'GCF_055389265.1', 'GCA_030035545.1', 'GCF_056784695.1', 'GCF_054903015.1', 'GCF_030035525.1', 'GCA_025252345.1', 'GCA_056784535.1', 'GCF_056790105.1', 'GCA_056784835.1', 'GCA_050956525.1', 'GCF_030035545.1', 'GCA_030295505.1', 'GCA_054903215.1', 'GCF_041235215.1', 'GCF_056784835.1', 'GCA_041233395.1', 'GCA_054902835.1', 'GCF_056784535.1', 'GCA_050956535.1', 'GCF_056784395.1', 'GCA_965645325.1', 'GCA_054903115.1', 'GCA_030295525.1', 'GCF_056784735.1', 'GCA_054902675.1', 'GCA_041053365.1', 'GCA_054903135.1', 'GCF_054902715.1', 'GCF_054902725.1', 'GCF_056784615.1', 'GCF_0567843

In [19]:
#filepath example --> 'data/FASTA/ecoli/PacBio/GCF_048571395.1/GCF_048571395.1_ASM4857139v1_genomic.fna'
prob_refs = []
for filepath in all_genome_paths:
    if '_genomic' not in filepath:
        #print(filepath)
        continue
    
    plasmid_output_path = os.path.join(os.path.dirname(filepath),
                                       os.path.basename(filepath).replace('_genomic', '_plasmid'))
    count = 0
    with open(plasmid_output_path, "w") as handle:
        writer = FastaWriter(handle, wrap=80)
        for record in SeqIO.parse(filepath, 'fasta'):
            if "plasmid" in record.description.lower():
                count += 1
                #handle.write(f">{record.description}\n{str(record.seq)}\n")
                writer.write_record(record)
    if count == 0:
        ref_id = filepath.split("/")[-2].strip()
        #if ref_id in new_refs:
        print(f"No plasmid found in {filepath}: {ref_id}")
        prob_refs.append(ref_id)
print(prob_refs)


No plasmid found in data/FinalFASTA/ecoli/PacBio/GCA_022570115.1/GCA_022570115.1_ASM2257011v1_genomic.fna: GCA_022570115.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCF_022569915.1/GCF_022569915.1_ASM2256991v1_genomic.fna: GCF_022569915.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCF_030345475.1/GCF_030345475.1_ASM3034547v1_genomic.fna: GCF_030345475.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCF_056784445.1/GCF_056784445.1_ASM5678444v1_genomic.fna: GCF_056784445.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCA_022569895.1/GCA_022569895.1_ASM2256989v1_genomic.fna: GCA_022569895.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCF_030345455.1/GCF_030345455.1_ASM3034545v1_genomic.fna: GCF_030345455.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCA_056784275.1/GCA_056784275.1_ASM5678427v1_genomic.fna: GCA_056784275.1
No plasmid found in data/FinalFASTA/ecoli/PacBio/GCF_030345555.1/GCF_030345555.1_ASM3034555v1_genomic.fna: GCF_030345555.1
No plasmid found